In [39]:
import sys
import os

# Get the absolute path of the parent directory
parent_dir = os.path.abspath('..')

# Add it to sys.path so Python can find the 'src' module
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Added {parent_dir} to Python path.")

Added /opt/spark to Python path.


**imports and Spark session**

In [40]:
from etl_pipeline.utils.spark_session import get_spark_session
from etl_pipeline.utils.s3_reader import read_delta_table

print("Imports successful")

Imports successful


In [41]:
spark = get_spark_session("SilverLayer-Orders")
print("Spark session ready")

Spark session ready


**list Bronze folder using Spark**

In [42]:
from config.settings import S3_BRONZE

# Create URI + Path
uri = spark._jvm.java.net.URI(S3_BRONZE)
path = spark._jvm.org.apache.hadoop.fs.Path(S3_BRONZE)

# Get correct filesystem based on s3a://
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    uri,
    spark._jsc.hadoopConfiguration()
)

# List all items in bronze
files = fs.listStatus(path)

for file in files:
    print(file.getPath().toString())

s3a://olist-lakehouse-2026/bronze/olist_customers_dataset
s3a://olist-lakehouse-2026/bronze/olist_geolocation_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_items_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_payments_dataset
s3a://olist-lakehouse-2026/bronze/olist_order_reviews_dataset
s3a://olist-lakehouse-2026/bronze/olist_orders_dataset
s3a://olist-lakehouse-2026/bronze/olist_products_dataset
s3a://olist-lakehouse-2026/bronze/olist_sellers_dataset
s3a://olist-lakehouse-2026/bronze/product_category_name_translation


# olist_orders_dataset

**read one Bronze dataset**

In [6]:
df_orders = read_delta_table(
    spark,
    "bronze",
    "olist_orders_dataset"
)

print("Bronze dataset loaded successfully")

Bronze dataset loaded successfully


**Preview sample rows**

In [6]:
df_orders.show(5, truncate=False, vertical=True)

26/04/12 15:35:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:>                                                          (0 + 1) / 1]

-RECORD 0---------------------------------------------------------
 order_id                      | 6ec1bea8cbcef0a1b81bc9b7fbd37ccb 
 customer_id                   | e6b5e20566e5c72cbaab04f91dec9c85 
 order_status                  | delivered                        
 order_purchase_timestamp      | 2018-08-07 05:42:46              
 order_approved_at             | 2018-08-07 05:50:08              
 order_delivered_carrier_date  | 2018-08-13 09:24:00              
 order_delivered_customer_date | 2018-08-27 15:28:22              
 order_estimated_delivery_date | 2018-09-18 00:00:00              
-RECORD 1---------------------------------------------------------
 order_id                      | 441972a5bbd51a10459a487402076942 
 customer_id                   | b79fa9dfed0c3d624b70fbd0ca2469de 
 order_status                  | delivered                        
 order_purchase_timestamp      | 2018-08-23 22:37:44              
 order_approved_at             | 2018-08-24 00:35:13          

**Inspect schema**

In [14]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



**Count all rows**

In [15]:
total_rows = df_orders.count()
print(f"Total rows: {total_rows}")

Total rows: 99441


**Check distinct business key**

In [16]:
distinct_orders = df_orders.select("order_id").distinct().count()
print(f"Distinct order_id: {distinct_orders}")

[Stage 19:===========================================>              (3 + 1) / 4]

Distinct order_id: 99441


**Count the total number of Null (missing) values in EACH COLUMN**

In [8]:
from pyspark.sql.functions import col, when, sum

df_orders.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_orders.columns
]).show()

[Stage 13:===========================================>              (3 + 1) / 4]

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [7]:
from pyspark.sql.functions import col, when, sum

df_orders.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_orders.columns
]).show(vertical=True)

26/04/15 09:57:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:============================================>              (3 + 1) / 4]

-RECORD 0-----------------------------
 order_id                      | 0    
 customer_id                   | 0    
 order_status                  | 0    
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



In [13]:
from pyspark.sql.functions import count, col

def spark_info(df):
    print("-" * 65)
    print(f"{'Tên Cột':<35} | {'Non-Null':<10} | {'Null':<10} | {'Dtype'}")
    print("-" * 65)
    
    # 1. Lấy tổng số dòng của DataFrame
    total_rows = df.count()
    
    # 2. Tính số lượng Non-Null cho tất cả các cột
    # Trong Spark, hàm count(col) tự động đếm các giá trị có thật và bỏ qua Null
    exprs = [count(col(c)).alias(c) for c in df.columns]
    non_null_counts = df.select(exprs).collect()[0].asDict()
    
    # 3. Lấy kiểu dữ liệu và in ra kết quả
    for col_name, dtype in df.dtypes:
        non_null = non_null_counts[col_name]
        null_count = total_rows - non_null
        
        # Format canh lề cho đẹp mắt
        print(f"{col_name:<35} | {non_null:<10} | {null_count:<10} | {dtype}")
        
    print("-" * 65)
    print(f"Tổng số dòng (Total Rows): {total_rows}")
    print(f"Tổng số cột (Total Columns): {len(df.columns)}")
    print("-" * 65)

spark_info(df_orders)

-----------------------------------------------------------------
Tên Cột                             | Non-Null   | Null       | Dtype
-----------------------------------------------------------------


[Stage 58:==============>                                           (1 + 3) / 4]

order_id                            | 99441      | 0          | string
customer_id                         | 99441      | 0          | string
order_status                        | 99441      | 0          | string
order_purchase_timestamp            | 99441      | 0          | timestamp
order_approved_at                   | 99281      | 160        | timestamp
order_delivered_carrier_date        | 97658      | 1783       | timestamp
order_delivered_customer_date       | 96476      | 2965       | timestamp
order_estimated_delivery_date       | 99441      | 0          | timestamp
-----------------------------------------------------------------
Tổng số dòng (Total Rows): 99441
Tổng số cột (Total Columns): 8
-----------------------------------------------------------------


**inspect order statuses**

In [19]:
df_orders.groupBy("order_status").count().show(truncate=False)

[Stage 32:==============>                                           (1 + 3) / 4]

+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |625  |
|approved    |2    |
|invoiced    |314  |
|delivered   |96478|
|unavailable |609  |
|processing  |301  |
|created     |5    |
+------------+-----+



**save the cleaned dataframe**

In [35]:
df_orders_clean = df_orders

**Save to Silver layer**

In [39]:
from etl_pipeline.utils.s3_writer import write_delta_table

write_delta_table(
    df_orders_clean,
    "silver",
    "siler_olist_orders_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/siler_olist_orders_dataset


**Important validation after save**

In [20]:
df_silver_test = read_delta_table(
    spark,
    "silver",
    "silver_olist_orders_dataset"
)

print(df_silver_test.count())
df_silver_test.show(5, vertical=True)
df_silver_test.printSchema()

99441


[Stage 48:>                                                         (0 + 1) / 1]

-RECORD 0---------------------------------------------
 order_id                      | 6ec1bea8cbcef0a1b... 
 customer_id                   | e6b5e20566e5c72cb... 
 order_status                  | delivered            
 order_purchase_timestamp      | 2018-08-07 05:42:46  
 order_approved_at             | 2018-08-07 05:50:08  
 order_delivered_carrier_date  | 2018-08-13 09:24:00  
 order_delivered_customer_date | 2018-08-27 15:28:22  
 order_estimated_delivery_date | 2018-09-18 00:00:00  
-RECORD 1---------------------------------------------
 order_id                      | 441972a5bbd51a104... 
 customer_id                   | b79fa9dfed0c3d624... 
 order_status                  | delivered            
 order_purchase_timestamp      | 2018-08-23 22:37:44  
 order_approved_at             | 2018-08-24 00:35:13  
 order_delivered_carrier_date  | 2018-08-24 13:19:00  
 order_delivered_customer_date | 2018-08-29 18:03:26  
 order_estimated_delivery_date | 2018-09-04 00:00:00  
-RECORD 2-

# olist_order_items_dataset

**1. Read dataset from Bronze**

In [22]:
df_order_items = read_delta_table(spark, "bronze", "olist_order_items_dataset")
print("Read succesfully")

Read succesfully


**2. Preview some rows**

In [22]:
df_order_items.show(5, truncate=False, vertical=True)

print(df_order_items.count())

-RECORD 0-----------------------------------------------
 order_id            | f40512005e99b84b0daf7b1ace9dbc7f 
 order_item_id       | 1                                
 product_id          | b623b7cb05ee3248fbe4a6ecbeed79a4 
 seller_id           | 7aa4334be125fcdd2ba64b3180029f14 
 shipping_limit_date | 2017-12-12 19:53:43              
 price               | 70.97                            
 freight_value       | 17.75                            
-RECORD 1-----------------------------------------------
 order_id            | f406a7c4fc4512aa60c7240e3973597e 
 order_item_id       | 1                                
 product_id          | cec09725da5ed01471d9a505e7389d37 
 seller_id           | 4d6d651bd7684af3fffabd5f08d12e5a 
 shipping_limit_date | 2017-07-13 14:23:36              
 price               | 69.9                             
 freight_value       | 16.12                            
-RECORD 2-----------------------------------------------
 order_id            | f407e662

**3. Print Schema**

In [53]:
df_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



**4. Check Nulls**

In [24]:
from pyspark.sql.functions import col, count, when

df_order_items.select([
    count(when(col(c).isNull(), True)).alias(c) 
    for c in df_order_items.columns
]).show()

[Stage 70:===========================================>              (3 + 1) / 4]

+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



In [23]:
spark_info(df_order_items)

-----------------------------------------------------------------
Tên Cột                             | Non-Null   | Null       | Dtype
-----------------------------------------------------------------


[Stage 106:==========================================>              (3 + 1) / 4]

order_id                            | 112650     | 0          | string
order_item_id                       | 112650     | 0          | int
product_id                          | 112650     | 0          | string
seller_id                           | 112650     | 0          | string
shipping_limit_date                 | 112650     | 0          | timestamp
price                               | 112650     | 0          | double
freight_value                       | 112650     | 0          | double
-----------------------------------------------------------------
Tổng số dòng (Total Rows): 112650
Tổng số cột (Total Columns): 7
-----------------------------------------------------------------


**5. Check unique**

In [55]:
from pyspark.sql.functions import col
df_order_items.groupBy("order_id", "order_item_id").count().filter(col("count") > 1).show()

[Stage 330:==========================================>              (3 + 1) / 4]

+--------+-------------+-----+
|order_id|order_item_id|count|
+--------+-------------+-----+
+--------+-------------+-----+



**6. Check constraint**

In [25]:
# Returns rows in df_child with foreign_key values NOT in df_parent
orphaned_records = df_order_items.join(df_silver_test, 
                                 df_order_items["order_id"] == df_silver_test["order_id"], 
                                 "left_anti")

if orphaned_records.isEmpty():
    print("Referential integrity maintained: All FK values exist in PK.")
else:
    print(f"Found {orphaned_records.count()} orphaned records.")
    orphaned_records.show()

[Stage 79:>                                                         (0 + 3) / 3]

Referential integrity maintained: All FK values exist in PK.


In [59]:
# Returns rows in df_child with foreign_key values NOT in df_parent
orphaned_records_1 = df_order_items.join(df_products_check, 
                                 df_order_items["product_id"] == df_products_check["product_id"], 
                                 "left_anti")

if orphaned_records_1.isEmpty():
    print("Referential integrity maintained: All FK values exist in PK.")
else:
    print(f"Found {orphaned_records_1.count()} orphaned records.")
    orphaned_records_1.show()

[Stage 376:======================================>                  (2 + 1) / 3]

Referential integrity maintained: All FK values exist in PK.


In [62]:
# Returns rows in df_child with foreign_key values NOT in df_parent
orphaned_records_2 = df_order_items.join(df_sellers_check, 
                                 df_order_items["seller_id"] == df_sellers_check["seller_id"], 
                                 "left_anti")

if orphaned_records_2.isEmpty():
    print("Referential integrity maintained: All FK values exist in PK.")
else:
    print(f"Found {orphaned_records_1.count()} orphaned records.")
    orphaned_records_2.show()

[Stage 391:>                                                        (0 + 3) / 3]

Referential integrity maintained: All FK values exist in PK.


**7. Check negative values**

In [63]:
from pyspark.sql import functions as F

# Count negative values in column 'A'
negative_count_price = df_order_items.filter(F.col("price") < 0).count()
print(f"Number of negative values in column price: {negative_count_price}")

negative_count_freight = df_order_items.filter(F.col("freight_value") < 0).count()
print(f"Number of negative values in column freight_value: {negative_count_freight}")

Number of negative values in column price: 0
Number of negative values in column freight_value: 0


**8. Save the cleaned DF then Save to Silver layer**

In [64]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_order_items_cleaned = df_order_items

write_delta_table(
    df_order_items_cleaned,
    "silver",
    "silver_olist_order_items_dataset"
)

print("Save successfully")

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_order_items_dataset
Save successfully


**9. Checking after save**

In [65]:
df_silver_order_items = read_delta_table(
    spark,
    "silver",
    "silver_olist_order_items_dataset"
)

print(df_silver_order_items.count())
df_silver_order_items.show(5)
df_silver_order_items.printSchema()

112650


[Stage 424:>                                                        (0 + 1) / 1]

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

# olist_customers_dataset

**1. Read dataset**

In [14]:
df_customers = read_delta_table(
    spark,
    "bronze",
    "olist_customers_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [32]:
df_customers.show(5, truncate=False)
df_customers.printSchema()
print("Total rows:", df_customers.count())

+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|
+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|50edf1bb01f965f14bee7527c3bed42f|58b5f3850e802b91a068ab89ba51686f|27195                   |santanesia   |RJ            |
|91489087a2f86dd3476f0d949946aea2|7b1af6c1d00a207d73873832fc6d18a4|6112                    |osasco       |SP            |
|0d2ff29c36a65d389738620aa2f4e145|8020cb61d67a964fdf5f246d5f6cb201|13418                   |piracicaba   |SP            |
|41e9548a0f8a774ff63837fa26f9839d|92ae32fae1d6d9fcfffd36717ae5166c|5019                    |sao paulo    |SP            |
|2acdf2f6aadbbfede01eccc71529cdb1|9bb28ed7c7fbd7e2392ed20b3fdd9818|96300                   |jaguarao     |RS            |
+-----------------------

**3. Check Nulls**

In [33]:
from pyspark.sql.functions import col, count, when

# Summarize null counts for all columns
df_customers.select([count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns]).show()

[Stage 185:==========================================>              (3 + 1) / 4]

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [15]:
spark_info(df_customers)

-----------------------------------------------------------------
Tên Cột                             | Non-Null   | Null       | Dtype
-----------------------------------------------------------------


[Stage 74:===================>                                      (1 + 2) / 3]

customer_id                         | 99441      | 0          | string
customer_unique_id                  | 99441      | 0          | string
customer_zip_code_prefix            | 99441      | 0          | int
customer_city                       | 99441      | 0          | string
customer_state                      | 99441      | 0          | string
-----------------------------------------------------------------
Tổng số dòng (Total Rows): 99441
Tổng số cột (Total Columns): 5
-----------------------------------------------------------------


**4. Check unique**

In [35]:
from pyspark.sql import functions as F

In [36]:
# Check for actual errors (same order AND same payment sequence)
actual_duplicates = (
    df_customers.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

actual_duplicates.show()

[Stage 190:==========================================>              (3 + 1) / 4]

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



**5. Check Distinct Values**

In [37]:
df_customers.select("customer_city").distinct().show()
df_customers.select("customer_state").distinct().show()

+--------------------+
|       customer_city|
+--------------------+
|            camacari|
|           itaberaba|
|           igrejinha|
|                iepe|
|            barracao|
|           arapiraca|
|                pote|
|divino de sao lou...|
|  aguas de sao pedro|
|jijoca de jericoa...|
|              bacaxa|
|   redencao da serra|
|           boa vista|
|            itanhaem|
|                ijui|
|             brusque|
|            perdigao|
|  cachoeira paulista|
|    leandro ferreira|
|           cachoeira|
+--------------------+
only showing top 20 rows



[Stage 202:==========================================>              (3 + 1) / 4]

+--------------+
|customer_state|
+--------------+
|            SC|
|            RO|
|            PI|
|            AM|
|            RR|
|            GO|
|            TO|
|            MT|
|            SP|
|            ES|
|            PB|
|            RS|
|            MS|
|            AL|
|            MG|
|            PA|
|            BA|
|            SE|
|            PE|
|            CE|
+--------------+
only showing top 20 rows



In [38]:
df_customers.select("customer_city").filter(
    col("customer_city").like("sao jose do rio preto")
).show()

[Stage 207:>                                                        (0 + 1) / 1]

+--------------------+
|       customer_city|
+--------------------+
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
|sao jose do rio p...|
+--------------------+
only showing top 20 rows



**6. Check key constraint**

In [39]:
from pyspark.sql.functions import col, lpad

# 1. Cast to string and pad with leading zeros up to 5 characters
df_customers = df_customers.withColumn(
    "customer_zip_code_prefix", 
    lpad(col("customer_zip_code_prefix").cast("string"), 5, "0")
)

# 2. Verify the fix
df_customers.filter(col("customer_zip_code_prefix") < "10000").show(5)

[Stage 210:>                                                        (0 + 1) / 1]

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|208954ebe4665c5b2...|dc36c19e6e8bdcb2d...|                   09812|sao bernardo do c...|            SP|
|c2e1d65c47178eac5...|d1dae3a5d39b12618...|                   09370|                maua|            SP|
|167be5b45cbb0ea5a...|e3301490c373db926...|                   08223|           sao paulo|            SP|
|972cd773a42450ed3...|d7140656936a64175...|                   02013|           sao paulo|            SP|
|7b2a039a8b7b57d48...|fbf1b27fc02af440c...|                   02846|           sao paulo|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows



In [42]:
df_customers.filter(col("customer_zip_code_prefix") > "99999").show(5)

[Stage 239:===================>                                     (1 + 2) / 3]

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
+-----------+------------------+------------------------+-------------+--------------+



In [40]:
# Returns rows in df_child with foreign_key values NOT in df_parent
orphaned_records = df_customers.join(df_geo_check, 
                                 df_customers["customer_zip_code_prefix"] == df_geo_check["geolocation_zip_code_prefix"], 
                                 "left_anti")

if orphaned_records.isEmpty():
    print("Referential integrity maintained: All FK values exist in PK.")
else:
    print(f"Found {orphaned_records.count()} orphaned records.")
    orphaned_records.show()

Found 278 orphaned records.


[Stage 230:>                                                        (0 + 1) / 1]

+--------------------+--------------------+------------------------+-------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------------+--------------+
|daf22b7353d8e5021...|bfe6b5289dd1cec94...|                   49870|              itabi|            SE|
|5885969ba920706be...|d714959800e106298...|                   72002|           brasilia|            DF|
|79298f6a872008117...|7ef7e4cd493353520...|                   13307|                itu|            SP|
|fbb076794786e1bd0...|217754b74cc79168d...|                   28617|      nova friburgo|            RJ|
|d5b00ed5152f1e795...|dc8a6cdc8426f5cee...|                   73402|           brasilia|            DF|
|626a0f18050903535...|615eb18afcde37c5e...|                   56327|          petrolina|            PE|
|4cc039b3cff6bee52...|28b73781996ecf6d9...|                   72

**5. Save in Silver**

In [35]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_customers_cleaned = df_customers

write_delta_table(
    df_customers_cleaned,
    "silver",
    "silver_olist_customers_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_customers_dataset


**6. Check**

In [36]:
df_silver_customers = read_delta_table(
    spark,
    "silver",
    "silver_olist_customers_dataset"
)

print(df_silver_customers.count())
df_silver_customers.show(5)
df_silver_customers.printSchema()

99443


[Stage 312:>                                                        (0 + 1) / 1]

+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|50edf1bb01f965f14...|58b5f3850e802b91a...|                   27195|   santanesia|            RJ|
|91489087a2f86dd34...|7b1af6c1d00a207d7...|                    6112|       osasco|            SP|
|0d2ff29c36a65d389...|8020cb61d67a964fd...|                   13418|   piracicaba|            SP|
|41e9548a0f8a774ff...|92ae32fae1d6d9fcf...|                    5019|    sao paulo|            SP|
|2acdf2f6aadbbfede...|9bb28ed7c7fbd7e23...|                   96300|     jaguarao|            RS|
+--------------------+--------------------+------------------------+-------------+--------------+
only showing top 5 rows

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = t

# olist_geolocation_dataset

**1. Read dataset**

In [16]:
df_geolocation = read_delta_table(
    spark,
    "bronze",
    "olist_geolocation_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [15]:
df_geolocation.show(5, truncate=False)
df_geolocation.printSchema()
print("Total rows: ", df_geolocation.count())

+---------------------------+------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat   |geolocation_lng    |geolocation_city|geolocation_state|
+---------------------------+------------------+-------------------+----------------+-----------------+
|50760                      |-8.072588074383969|-34.91359211518264 |recife          |PE               |
|50740                      |-8.042627210658422|-34.946004300305844|recife          |PE               |
|50751                      |-8.066622635581963|-34.917061201223355|recife          |PE               |
|50731                      |-8.038528009435058|-34.93686693228113 |recife          |PE               |
|50720                      |-8.056531245917173|-34.91027403912522 |recife          |PE               |
+---------------------------+------------------+-------------------+----------------+-----------------+
only showing top 5 rows

root
 |-- geolocation_zip_code_prefix: 

In [17]:
spark_info(df_geolocation)

-----------------------------------------------------------------
Tên Cột                             | Non-Null   | Null       | Dtype
-----------------------------------------------------------------


[Stage 90:=================================================>        (6 + 1) / 7]

geolocation_zip_code_prefix         | 1000163    | 0          | int
geolocation_lat                     | 1000163    | 0          | double
geolocation_lng                     | 1000163    | 0          | double
geolocation_city                    | 1000163    | 0          | string
geolocation_state                   | 1000163    | 0          | string
-----------------------------------------------------------------
Tổng số dòng (Total Rows): 1000163
Tổng số cột (Total Columns): 5
-----------------------------------------------------------------


**3. Check duplicate**

In [20]:
from pyspark.sql.functions import col
df_geolocation.groupBy("geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng").count().filter(col("count") > 1).show()

[Stage 100:==================================================>    (11 + 1) / 12]

+---------------------------+------------------+-------------------+-----+
|geolocation_zip_code_prefix|   geolocation_lat|    geolocation_lng|count|
+---------------------------+------------------+-------------------+-----+
|                      50710|-8.045672858213726| -34.90429851708594|    5|
|                      51020|-8.117835872612345| -34.90021233680533|    5|
|                      51020|-8.119364919787646|-34.898912633360254|    2|
|                      51030|-8.140673724392665| -34.90787678449414|    2|
|                      52021| -8.04240848681403|-34.890185409660084|    2|
|                      53170|-7.987534078776274|  -34.9059736221188|    2|
|                      53560|-7.904034785715728| -34.90947519008841|    2|
|                      56170|-8.619122746725441| -39.60227892937364|    4|
|                      57085|-9.558689606614918|  -35.7422678638538|    2|
|                      57025|-9.672862955594647| -35.75816874216626|    2|
|                      57

In [21]:
# Drop rows that have the exact same zip code AND coordinates
df_geolocation = df_geolocation.dropDuplicates([
    "geolocation_zip_code_prefix", 
    "geolocation_lat", 
    "geolocation_lng"
])

# Optional: You can also just use df_geolocation.dropDuplicates() with no arguments 
# to drop rows where EVERY single column (including city/state) is identical.

**4. Check Nulls**

In [22]:
from pyspark.sql.functions import col, count, when

df_geolocation.select([count(when(col(c).isNull(), c)).alias(c) for c in df_geolocation.columns]).show()

[Stage 105:==================================================>    (11 + 1) / 12]

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



**5. Check type of zip code**

In [27]:
df_geolocation.filter(df_geolocation["geolocation_zip_code_prefix"] < 10000).show()

[Stage 140:======================================>                  (2 + 1) / 3]

+---------------------------+-------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+----------------+-----------------+
|                       1001|-23.550497706907514| -46.63433817805407|       sao paulo|               SP|
|                       1001|-23.550263371631395| -46.63419639384839|       são paulo|               SP|
|                       1001|-23.549951273933896| -46.63402708563671|       são paulo|               SP|
|                       1001|-23.549779299469115|  -46.6339571183853|       são paulo|               SP|
|                       1003|-23.548900652778652| -46.63715701234924|       sao paulo|               SP|
|                       1004| -23.55046690313923| -46.63523313973692|       sao paulo|               SP|
|                       1004|-23.550202089647414| -46.6

In [28]:
from pyspark.sql.functions import col, lpad

# 1. Cast to string and pad with leading zeros up to 5 characters
df_geolocation = df_geolocation.withColumn(
    "geolocation_zip_code_prefix", 
    lpad(col("geolocation_zip_code_prefix").cast("string"), 5, "0")
)

# 2. Verify the fix
df_geolocation.filter(col("geolocation_zip_code_prefix") < "10000").show(5)

[Stage 145:==================================================>    (11 + 1) / 12]

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                      01001|-23.550497706907514|-46.63433817805407|       sao paulo|               SP|
|                      01001|-23.550263371631395|-46.63419639384839|       são paulo|               SP|
|                      01001|-23.549951273933896|-46.63402708563671|       são paulo|               SP|
|                      01001|-23.549779299469115| -46.6339571183853|       são paulo|               SP|
|                      01003|-23.548900652778652|-46.63715701234924|       sao paulo|               SP|
+---------------------------+-------------------+------------------+----------------+-----------------+
only showing top 5 rows



In [41]:
df_geolocation.filter(col("geolocation_zip_code_prefix") > "99999").show(5)

[Stage 233:==================================================>    (11 + 1) / 12]

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
+---------------------------+---------------+---------------+----------------+-----------------+



**6. Save in Silver**

In [29]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_geolocation_cleaned = df_geolocation

write_delta_table(
    df_geolocation_cleaned,
    "silver",
    "silver_olist_geolocation_dataset"
)

26/04/07 12:33:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/07 12:33:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/07 12:33:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/07 12:33:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/07 12:33:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/07 12:33:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/07 12:33:35 WARN MemoryManager: Total allocation exceeds 95.

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_geolocation_dataset


**7. Verify the output**

In [30]:
df_geo_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_geolocation_dataset"
)

df_geo_check.printSchema()
df_geo_check.show(5)
print("Total rows:", df_geo_check.count())

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



+---------------------------+-------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+----------------+-----------------+
|                      01001| -23.55064182209015|-46.634409790322515|       sao paulo|               SP|
|                      01003| -23.54904445268046| -46.63518261226844|       sao paulo|               SP|
|                      01003|  -23.5489584902063| -46.63486176165939|       sao paulo|               SP|
|                      01006|-23.550317224536894| -46.63660363835193|       são paulo|               SP|
|                      01011|-23.545577325841062| -46.63477539133823|       sao paulo|               SP|
+---------------------------+-------------------+-------------------+----------------+-----------------+
only showing top 5 rows

Total rows: 720154


# olist_order_payments_dataset

**1. Read dataset**

In [24]:
df_order_payments = read_delta_table(
    spark,
    "bronze",
    "olist_order_payments_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [5]:
df_order_payments.printSchema()
df_order_payments.show(5, truncate=False)
print("Total rows:", df_order_payments.count())

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



26/04/07 06:05:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|e558bf8bb8fddae947c5ab3a53b8ff64|1                 |credit_card |1                   |371.13       |
|2095778434755b568ba4c3c2325096fe|1                 |credit_card |2                   |55.11        |
|5c1995c020e0a3b2b23152b3fcfcda76|1                 |credit_card |1                   |165.8        |
|ab459ccd2e24ef6d6390517c8b9809a4|1                 |boleto      |1                   |188.62       |
|9b3d13c2644e10e20020ccd7f3d40534|1                 |credit_card |5                   |65.71        |
+--------------------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows

Total rows: 103889


In [25]:
spark_info(df_order_payments)

-----------------------------------------------------------------
Tên Cột                             | Non-Null   | Null       | Dtype
-----------------------------------------------------------------


[Stage 122:>                                                        (0 + 2) / 2]

order_id                            | 103886     | 0          | string
payment_sequential                  | 103886     | 0          | int
payment_type                        | 103886     | 0          | string
payment_installments                | 103886     | 0          | int
payment_value                       | 103886     | 0          | double
-----------------------------------------------------------------
Tổng số dòng (Total Rows): 103886
Tổng số cột (Total Columns): 5
-----------------------------------------------------------------


**3. Check unique**

In [12]:
# Check for actual errors (same order AND same payment sequence)
actual_duplicates = (
    df_order_payments.groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
)

actual_duplicates.show()

[Stage 95:======================================>                   (2 + 1) / 3]

+--------+------------------+-----+
|order_id|payment_sequential|count|
+--------+------------------+-----+
+--------+------------------+-----+



**4. Check Nulls**

In [14]:
from pyspark.sql.functions import col, count, when

df_order_payments.select([count(when(col(c).isNull(), c)).alias(c) for c in df_order_payments.columns]).show()

[Stage 107:======================================>                  (2 + 1) / 3]

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



**5. Check key constraint**

In [15]:
df_silver_olist_orders = read_delta_table(
    spark,
    "silver",
    "silver_olist_orders_dataset"
)

# Returns rows in 'orders' (FK table) where 'customer_id' is NOT in 'customers' (PK table)
orphaned_rows = df_order_payments.join(df_silver_olist_orders, 
                               df_order_payments.order_id == df_silver_olist_orders.order_id, 
                               how='left_anti')

if orphaned_rows.count() > 0:
    print("Referential integrity violation: Missing keys found.")
    orphaned_rows.show()
    print(orphaned_rows.count())

Referential integrity violation: Missing keys found.


+--------------+------------------+------------+--------------------+-------------+
|      order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------+------------------+------------+--------------------+-------------+
|demo_order_001|                 1| credit_card|                   3|        299.9|
|demo_order_002|                 1|      boleto|                   1|        150.0|
|demo_order_003|                 1|  debit_card|                   1|         89.9|
+--------------+------------------+------------+--------------------+-------------+



[Stage 130:======================================>                  (2 + 1) / 3]

3


In [17]:
# 1. Load the clean orders table (if you haven't already in this cell)
# df_orders = read_delta_table(spark, "silver", "silver_olist_orders_dataset")

# 2. Keep ONLY reviews where the order_id exists in the clean orders table
df_order_payments = df_order_payments.join(
    df_silver_olist_orders,
    df_order_payments["order_id"] == df_silver_olist_orders["order_id"],
    "left_semi"
)

# 3. Verify they are gone
print(f"Cleaned reviews count: {df_order_payments.count()}")

[Stage 138:===================>                                     (1 + 2) / 3]

Cleaned reviews count: 103886


**6. Distinct values**

In [18]:
df_order_payments.select("payment_type").distinct().show()
print(df_order_payments.count())

+------------+
|payment_type|
+------------+
|      boleto|
| not_defined|
| credit_card|
|     voucher|
|  debit_card|
+------------+



[Stage 154:======================================>                  (2 + 1) / 3]

103886


In [19]:
payment_mapping = {
    "credit_card": "credit card",
    "debit_card": "debit card",
    "not_defined": "not defined"
}

df_order_payments = df_order_payments.replace(
    payment_mapping,
    subset=["payment_type"]
)

In [20]:
df_order_payments.select("payment_type").distinct().show()
print(df_order_payments.count())

+------------+
|payment_type|
+------------+
|  debit card|
|      boleto|
|     voucher|
| credit card|
| not defined|
+------------+



[Stage 170:======================================>                  (2 + 1) / 3]

103886


**6. Check negative values**

In [21]:
from pyspark.sql import functions as F

# List of numeric columns to check
cols_to_check = ["payment_sequential", "payment_installments", "payment_value"]

# Create count expressions for each column
count_exprs = [F.count(F.when(F.col(c) < 0, 1)).alias(c) for c in cols_to_check]

df_order_payments.select(count_exprs).show()

[Stage 178:======================================>                  (2 + 1) / 3]

+------------------+--------------------+-------------+
|payment_sequential|payment_installments|payment_value|
+------------------+--------------------+-------------+
|                 0|                   0|            0|
+------------------+--------------------+-------------+



**7. Save in silver**

In [22]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_order_payments_cleaned = df_order_payments

write_delta_table(
    df_order_payments_cleaned,
    "silver",
    "silver_olist_order_payments_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_order_payments_dataset


**8. Verify output**

In [23]:
df_order_payments_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_order_payments_dataset"
)

df_order_payments_check.printSchema()
df_order_payments_check.show(5)
print("Total rows:", df_order_payments_check.count())

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|e558bf8bb8fddae94...|                 1| credit card|                   1|       371.13|
|2095778434755b568...|                 1| credit card|                   2|        55.11|
|5c1995c020e0a3b2b...|                 1| credit card|                   1|        165.8|
|ab459ccd2e24ef6d6...|                 1|      boleto|                   1|       188.62|
|9b3d13c2644e10e20...|                 1| credit card|                   5|        65.71|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows

Total rows: 103886


# olist_order_reviews_dataset

**1. Read dataset**

In [45]:
df_order_reviews = read_delta_table(
    spark,
    "bronze",
    "olist_order_reviews_dataset"
)

print("Read successfully")

Read successfully


**2. Preview dataset**

In [44]:
df_order_reviews.printSchema()
df_order_reviews.show(5, truncate=False)
print("Toal rows: ", df_order_reviews.count())

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                                                                                                |2018-01-18 00:00:00 |2018-01-18 21:46:59    |
|80e641a

**4. Check nulls**

In [45]:
from pyspark.sql.functions import col, sum

# Returns a count of nulls for every column
df_order_reviews.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_order_reviews.columns]).show()

[Stage 287:==========================================>              (3 + 1) / 4]

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [46]:
spark_info(df_order_reviews)

-----------------------------------------------------------------
Tên Cột                             | Non-Null   | Null       | Dtype
-----------------------------------------------------------------


[Stage 210:======================================>                  (2 + 1) / 3]

review_id                           | 99224      | 0          | string
order_id                            | 99224      | 0          | string
review_score                        | 99224      | 0          | int
review_comment_title                | 11568      | 87656      | string
review_comment_message              | 40977      | 58247      | string
review_creation_date                | 99224      | 0          | timestamp
review_answer_timestamp             | 99224      | 0          | timestamp
-----------------------------------------------------------------
Tổng số dòng (Total Rows): 99224
Tổng số cột (Total Columns): 7
-----------------------------------------------------------------


In [46]:
# Drop rows where the primary key OR the foreign key is null
df_order_reviews = df_order_reviews.dropna(subset=["review_id", "order_id"])

# Returns a count of nulls for every column
df_order_reviews.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_order_reviews.columns]).show()

# Verify they are gone
print(f"Total rows after dropping null IDs: {df_order_reviews.count()}")

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        0|       0|         144|               89922|                 60843|                6528|                   6548|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



[Stage 297:==========================================>              (3 + 1) / 4]

Total rows after dropping null IDs: 101925


**5. Check duplicate**

In [47]:
from pyspark.sql import functions as F
df_order_reviews.groupBy("review_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .show()

[Stage 302:==========================================>              (3 + 1) / 4]

+--------------------+-----+
|           review_id|count|
+--------------------+-----+
|868c4827b9ab72723...|    2|
|a8ffb30c74397c7cf...|    2|
|f144ac19984746532...|    2|
|e4f5fbbbf2fa8f259...|    2|
|f6e05a3f6dbc3bdb1...|    2|
|c9a406d3076559e35...|    2|
|dd43a798c9e8baf7b...|    2|
|c1dd7c6a2154caaf1...|    2|
|bee3c76b6017ec9de...|    2|
|f45e3373e177f5694...|    2|
|1961cafe1d1cbd4cc...|    2|
|090aa2980b0aea78c...|    2|
|59d160f8ce9217282...|    2|
|f3544b47fdfb4bdd2...|    2|
|9840563f4c2189d0a...|    2|
|8b8f495d30fca09fd...|    2|
|1c7b8b15de9a2eff8...|    2|
|0a2d1206f8b2473cd...|    2|
|a46a4abee1cb1ec1c...|    2|
|30153632ebb19a673...|    2|
+--------------------+-----+
only showing top 20 rows



In [48]:
# Drop duplicates based on the 'review_id' column
df_order_reviews = df_order_reviews.dropDuplicates(["review_id"])

# Verify they are gone
print(f"Total rows after deduplication: {df_order_reviews.count()}")

[Stage 307:==========================================>              (3 + 1) / 4]

Total rows after deduplication: 100789


**6. Check validate date**

In [49]:
from pyspark.sql.functions import col, to_date
# Ensure columns are cast to date
df_order_reviews.filter(to_date(col("review_creation_date")) > to_date(col("review_answer_timestamp"))).show()

[Stage 315:==========================================>              (3 + 1) / 4]

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



**7. Check negative values**

In [50]:
from pyspark.sql import functions as F

# Filter rows where 'col_name' is less than 0
df_order_reviews.filter(F.col("review_score") < 0).show()

[Stage 324:==========================================>              (3 + 1) / 4]

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



**8. Convert data type**

In [53]:
from pyspark.sql.functions import to_date, to_timestamp, col

df_order_reviews = df_order_reviews.withColumn(
    "review_creation_date",
    to_timestamp(col("review_creation_date"))
)

df_order_reviews = df_order_reviews.withColumn(
    "review_answer_timestamp",
    to_timestamp(col("review_answer_timestamp"))
)

df_order_reviews = df_order_reviews.withColumn(
    "review_score",
    col("review_score").cast("int")
)

In [54]:
df_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [55]:
df_order_reviews.groupBy("review_score").count().show()

[Stage 333:==========================================>              (3 + 1) / 4]

+------------+-----+
|review_score|count|
+------------+-----+
|        NULL| 2376|
|           1|11282|
|           3| 8097|
|           5|56910|
|           4|19007|
|           2| 3114|
|          90|    1|
|           0|    2|
+------------+-----+



**9. Remove outlier**

In [56]:
from pyspark.sql.functions import col

df_order_reviews = df_order_reviews.filter(
    col("review_score").isNull() |
    col("review_score").between(1, 5)
)

In [57]:
df_order_reviews.groupBy("review_score").count().show()

[Stage 341:==========================================>              (3 + 1) / 4]

+------------+-----+
|review_score|count|
+------------+-----+
|        NULL| 2376|
|           1|11282|
|           3| 8097|
|           5|56910|
|           4|19007|
|           2| 3114|
+------------+-----+



**10. Check key constraint**

In [58]:
# Returns rows in 'child_df' where 'fk_column' is NOT in 'parent_df.pk_column'
orphaned_records = df_order_reviews.join(df_orders, 
                                df_order_reviews["order_id"] == df_orders["order_id"], 
                                "left_anti")

if orphaned_records.count() > 0:
    print("Referential integrity violation found!")
    orphaned_records.show()
    print(orphaned_records.count())

Referential integrity violation found!


+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
| Ao se colocar o ...|            na CAPA |        NULL|        o Microfone |   ficam Bloquados ...| 2018-04-04 00:00:00|    2018-04-06 16:36:40|
| Gostaria que a e...| 2017-09-24 00:00:00|        NULL|                NULL|                  NULL|                NULL|                   NULL|
| Mas chegou dentr...| 2018-08-10 00:00:00|        NULL|                NULL|                  NULL|                NULL|                   NULL|
| Que arrependimen...| 2017-12-13 00:00:00|        NULL|                NULL|                  NULL|                NULL|   

2376


In [59]:
# 1. Load the clean orders table (if you haven't already in this cell)
# df_orders = read_delta_table(spark, "silver", "silver_olist_orders_dataset")

# 2. Keep ONLY reviews where the order_id exists in the clean orders table
df_order_reviews = df_order_reviews.join(
    df_silver_test,
    df_order_reviews["order_id"] == df_silver_test["order_id"],
    "left_semi"
)

# 3. Verify they are gone
print(f"Cleaned reviews count: {df_order_reviews.count()}")

Cleaned reviews count: 98410


**11. Save in silver**

In [60]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_order_reviews_cleaned = df_order_reviews

write_delta_table(
    df_order_reviews_cleaned,
    "silver",
    "silver_olist_order_reviews_dataset"
)

26/04/07 03:38:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/07 03:38:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/07 03:38:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/07 03:38:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/07 03:38:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/07 03:38:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/07 03:38:08 WARN MemoryManager: Total allocation exceeds 95.

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_order_reviews_dataset


**12. Verify output**

In [61]:
df_order_reviews_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_order_reviews_dataset"
)

df_order_reviews_check.printSchema()
df_order_reviews_check.show(5)
print("Total rows:", df_order_reviews_check.count())

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|00041b1f51e77a663...|998b1beee068744ac...|           1|                NULL|                  NULL| 2018-05-24 00:00:00|    2018-05-24 12:45:38|
|001736c33bb2a7a24...|df75a2d753c65af66...|           3|                NULL|  A entrega foi rap...| 2018-06-15 00:00:00|    2018-06-15 19:20:57|
|002e8e05b26cca7ce...|07220c39eb2606836...|           4|                NULL|     Aguardo o produto| 2018-04-11 00:00:00|    2018-04-11 11:09:23|
|003c15c7ac83b5e12...|e76180345fb828eab...|           5|                NULL|                  NULL| 2018-05-17 00:00:00|   

# olist_products_dataset

**1. Read dataset**

In [4]:
df_products = read_delta_table(
    spark,
    "bronze",
    "olist_products_dataset"
)

print("Bronze dataset loaded successfully")

26/04/07 11:51:51 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Bronze dataset loaded successfully


**2. Preview dataset**

In [5]:
df_products.printSchema()
df_products.show(5, truncate=False)
print("Total rows: ", df_products.count())

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



26/04/07 11:52:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                 |287                       |1                 |225             |16               |10               |14              |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                 |276                       |1                 |1000            |30               |18               |20              |
|96bd76ec8810374ed1b65e291975717f|e

**3. Check unique**

In [6]:
from pyspark.sql.functions import col
df_products.groupBy("product_id").count().filter(col("count") > 1).show()

[Stage 16:>                                                         (0 + 1) / 1]

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



**4. Check Nulls**

In [7]:
from pyspark.sql.functions import col, count, when

# Counts nulls for every column in the DataFrame
df_products.select([count(when(col(c).isNull(), c)).alias(c) for c in df_products.columns]).show()

[Stage 21:>                                                         (0 + 1) / 1]

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



**5. Check negative values**

In [8]:
from pyspark.sql.functions import col, count, when

numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

df_products.select([
    count(
        when(col(c) < 0, c)
    ).alias(c)
    for c in numeric_cols
]).show()

[Stage 26:>                                                         (0 + 1) / 1]

+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|                  0|                         0|                 0|               0|                0|                0|               0|
+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



**6. Check Distinct values**

In [9]:
df_products.groupBy("product_category_name").count().show()

[Stage 31:>                                                         (0 + 1) / 1]

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
|                  pcs|   30|
|                bebes|  919|
|                artes|   55|
|            cine_foto|   28|
|     moveis_decoracao| 2657|
|             pc_gamer|    3|
| construcao_ferram...|  400|
| tablets_impressao...|    9|
| fashion_roupa_mas...|   95|
|    artigos_de_festas|   26|
|     artigos_de_natal|   65|
|           la_cuisine|   10|
|               flores|   14|
|      livros_tecnicos|  123|
|                 NULL|  610|
|       telefonia_fixa|  116|
| construcao_ferram...|   91|
|           cool_stuff|  789|
|     eletrodomesticos|  370|
|    livros_importados|   31|
+---------------------+-----+
only showing top 20 rows



In [10]:
from pyspark.sql.functions import col, regexp_replace

df_products = df_products.withColumn(
    "product_category_name",
    regexp_replace(col("product_category_name"), "_", " ")
)

In [11]:
df_products.groupBy("product_category_name").count().show()

[Stage 36:>                                                         (0 + 1) / 1]

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
| construcao ferram...|  400|
|                  pcs|   30|
|                bebes|  919|
|                artes|   55|
| fashion bolsas e ...|  849|
|     artigos de natal|   65|
| portateis cozinha...|   10|
| sinalizacao e seg...|   93|
| tablets impressao...|    9|
|               flores|   14|
|    alimentos bebidas|  104|
|         beleza saude| 2444|
| instrumentos musi...|  289|
|                 NULL|  610|
| portateis casa fo...|   31|
| fashion roupa inf...|    5|
|     malas acessorios|  349|
|            cine foto|   28|
|     eletrodomesticos|  370|
| construcao ferram...|   39|
+---------------------+-----+
only showing top 20 rows



**6. Save in silver**

In [12]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_products_cleaned = df_products

write_delta_table(
    df_products_cleaned,
    "silver",
    "silver_olist_products_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_products_dataset


**7. Verify output**

In [58]:
df_products_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_products_dataset"
)

df_products_check.printSchema()
df_products_check.show(5)
print("Total rows:", df_products_check.count())

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte lazer|                 46|                       250|    

# olist_sellers_dataset

**1. Read dataset**

In [43]:
df_sellers = read_delta_table(
    spark,
    "bronze",
    "olist_sellers_dataset"
)

print("success")

success


**2. Preview dataset**

In [44]:
df_sellers.printSchema()
df_sellers.show(5, truncate=False)
print("Total rows:", df_sellers.count())

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



+--------------------------------+----------------------+-----------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|
+--------------------------------+----------------------+-----------------+------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |rio de janeiro   |RJ          |
|c0f3eea2e14555b6faeea3dd58c1b1c3|4195                  |sao paulo        |SP          |
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |braganca paulista|SP          |
+--------------------------------+----------------------+-----------------+------------+
only showing top 5 rows

Total rows: 3095


**3. Check Nulls**

In [45]:
from pyspark.sql.functions import col, count, when

# Counts nulls for every column in the DataFrame
df_sellers.select([count(when(col(c).isNull(), c)).alias(c) for c in df_sellers.columns]).show()

[Stage 256:>                                                        (0 + 1) / 1]

+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



**4. Check duplicate**

In [46]:
from pyspark.sql.functions import col
df_sellers.groupBy("seller_id").count().filter(col("count") > 1).show()

[Stage 261:>                                                        (0 + 1) / 1]

+---------+-----+
|seller_id|count|
+---------+-----+
+---------+-----+



**5. Check constraint**

In [48]:
from pyspark.sql.functions import col, lpad

# 1. Ensure BOTH tables have the zip code formatted as a 5-character string
df_sellers = df_sellers.withColumn(
    "seller_zip_code_prefix", 
    lpad(col("seller_zip_code_prefix").cast("string"), 5, "0")
)

# 2. Run your exact test
orphaned_records = df_sellers.join(
    df_geo_check, 
    df_sellers["seller_zip_code_prefix"] == df_geo_check["geolocation_zip_code_prefix"], 
    "left_anti"
)

if orphaned_records.isEmpty():
    print("Referential integrity maintained: All FK values exist in PK.")
else:
    print(f"Found {orphaned_records.count()} orphaned records.")
    orphaned_records.show()

Found 7 orphaned records.


[Stage 283:>                                                        (0 + 1) / 1]

+--------------------+----------------------+---------------+------------+
|           seller_id|seller_zip_code_prefix|    seller_city|seller_state|
+--------------------+----------------------+---------------+------------+
|5962468f885ea01a1...|                 82040|       curitiba|          PR|
|2aafae69bf4c41fbd...|                 91901|   porto alegre|          RS|
|2a50b7ee5aebecc6f...|                 72580|       brasilia|          DF|
|2e90cb1677d35cfe2...|                 02285|      sao paulo|          SP|
|0b3f27369a4d8df98...|                 07412|          aruja|          SP|
|42bde9fef835393bb...|                 71551|       brasilia|          DF|
|870d0118f7a9d8596...|                 37708|pocos de caldas|          MG|
+--------------------+----------------------+---------------+------------+



**6. Check distinct**

In [19]:
df_sellers.groupBy("seller_city").count().show()
df_sellers.groupBy("seller_state").count().show()

+--------------------+-----+
|         seller_city|count|
+--------------------+-----+
|           igrejinha|    1|
|             brusque|   10|
|            buritama|    1|
|         carapicuiba|   10|
|               garca|    4|
|  sao joao de meriti|    3|
|    fernando prestes|    1|
|             ipaussu|    1|
|           jacutinga|    3|
|       nova friburgo|    5|
|              araras|    3|
| sao pedro da aldeia|    1|
|              santos|   16|
|itapecerica da serra|    4|
|            ibitinga|   49|
|               muqui|    1|
|              cuiaba|    2|
|     franco da rocha|    2|
|               cotia|   10|
|             marilia|   15|
+--------------------+-----+
only showing top 20 rows



[Stage 93:>                                                         (0 + 1) / 1]

+------------+-----+
|seller_state|count|
+------------+-----+
|          SC|  190|
|          RO|    2|
|          PI|    1|
|          AM|    1|
|          GO|   40|
|          MT|    4|
|          SP| 1849|
|          PB|    6|
|          ES|   23|
|          RS|  129|
|          MS|    5|
|          MG|  244|
|          PA|    1|
|          BA|   19|
|          SE|    2|
|          PE|    9|
|          CE|   13|
|          RN|    5|
|          RJ|  171|
|          MA|    1|
+------------+-----+
only showing top 20 rows



**7. Save in silver**

In [49]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_sellers_cleaned = df_sellers

write_delta_table(
    df_sellers_cleaned,
    "silver",
    "silver_olist_sellers_dataset"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_olist_sellers_dataset


**8. Verify output**

In [60]:
df_sellers_check = read_delta_table(
    spark,
    "silver",
    "silver_olist_sellers_dataset"
)

df_sellers_check.printSchema()
df_sellers_check.show(5)
print("Total rows:", df_sellers_check.count())

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                 04195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
+--------------------+----------------------+-----------------+------------+
only showing top 5 rows

Total rows: 3095


# product_category_name_translation

**1. Read dataset**

In [22]:
df_product_category_name_translation = read_delta_table(
    spark,
    "bronze",
    "product_category_name_translation"
)

print("success")

success


**2. Preview dataset**

In [23]:
df_product_category_name_translation.printSchema()
df_product_category_name_translation.show(5, truncate=False)
print("Total rows:", df_product_category_name_translation.count())

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



+----------------------+-----------------------------+
|product_category_name |product_category_name_english|
+----------------------+-----------------------------+
|beleza_saude          |health_beauty                |
|informatica_acessorios|computers_accessories        |
|automotivo            |auto                         |
|cama_mesa_banho       |bed_bath_table               |
|moveis_decoracao      |furniture_decor              |
+----------------------+-----------------------------+
only showing top 5 rows

Total rows: 71


**3. Check nulls**

In [24]:
from pyspark.sql.functions import col, count, when

# Counts nulls for every column in the DataFrame
df_product_category_name_translation.select([count(when(col(c).isNull(), c)).alias(c) for c in df_product_category_name_translation.columns]).show()

[Stage 129:>                                                        (0 + 1) / 1]

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



**4. Name friendly**

In [25]:
from pyspark.sql.functions import col, regexp_replace

columns_to_clean = [
    "product_category_name",
    "product_category_name_english"
]

for c in columns_to_clean:
    df_product_category_name_translation = df_product_category_name_translation.withColumn(
        c,
        regexp_replace(col(c), "_", " ")
    )

In [26]:
df_product_category_name_translation.groupBy("product_category_name").count().show()
df_product_category_name_translation.groupBy("product_category_name_english").count().show()

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
| construcao ferram...|    1|
|                  pcs|    1|
|                bebes|    1|
|                artes|    1|
| fashion bolsas e ...|    1|
|     artigos de natal|    1|
| sinalizacao e seg...|    1|
| tablets impressao...|    1|
|               flores|    1|
|    alimentos bebidas|    1|
|         beleza saude|    1|
| instrumentos musi...|    1|
| portateis casa fo...|    1|
| fashion roupa inf...|    1|
|     malas acessorios|    1|
|            cine foto|    1|
|     eletrodomesticos|    1|
| construcao ferram...|    1|
|    livros importados|    1|
| fashion roupa mas...|    1|
+---------------------+-----+
only showing top 20 rows



[Stage 139:>                                                        (0 + 1) / 1]

+-----------------------------+-----+
|product_category_name_english|count|
+-----------------------------+-----+
|                          art|    1|
|          musical instruments|    1|
|                      flowers|    1|
|          diapers and hygiene|    1|
|         fashion bags acce...|    1|
|             office furniture|    1|
|                 garden tools|    1|
|              furniture decor|    1|
|                    computers|    1|
|         industry commerce...|    1|
|                         auto|    1|
|         fashion childrens...|    1|
|           christmas supplies|    1|
|         signaling and sec...|    1|
|              home appliances|    1|
|         costruction tools...|    1|
|         fashion underwear...|    1|
|                         food|    1|
|         books general int...|    1|
|         construction tool...|    1|
+-----------------------------+-----+
only showing top 20 rows



**5. Save in silver**

In [27]:
from etl_pipeline.utils.s3_writer import write_delta_table

df_translation_cleaned = df_product_category_name_translation

write_delta_table(
    df_translation_cleaned,
    "silver",
    "silver_product_category_name_translation"
)

Saved successfully to s3a://olist-lakehouse-2026/silver/silver_product_category_name_translation


**6. Verify output**

In [5]:
df_translation_check = read_delta_table(
    spark,
    "silver",
    "silver_product_category_name_translation"
)

df_translation_check.printSchema()
df_translation_check.show(5)
print("Total rows:", df_translation_check.count())

26/04/06 12:17:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



26/04/06 12:17:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza saude|                health beauty|
| informatica acess...|         computers accesso...|
|           automotivo|                         auto|
|      cama mesa banho|               bed bath table|
|     moveis decoracao|              furniture decor|
+---------------------+-----------------------------+
only showing top 5 rows

Total rows: 71


**List silver dataset**

In [66]:
def list_layer_datasets(spark, layer_path):
    uri = spark._jvm.java.net.URI(layer_path)
    path = spark._jvm.org.apache.hadoop.fs.Path(layer_path)

    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        uri,
        spark._jsc.hadoopConfiguration()
    )

    files = fs.listStatus(path)

    return [file.getPath().toString() for file in files]

In [67]:
from config.settings import S3_SILVER

silver_tables = list_layer_datasets(spark, S3_SILVER)

for table in silver_tables:
    print(table)

s3a://olist-lakehouse-2026/silver/olist_orders_dataset
s3a://olist-lakehouse-2026/silver/siler_olist_orders_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_customers_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_geolocation_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_items_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_payments_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_reviews_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_orders_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_products_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_sellers_dataset
s3a://olist-lakehouse-2026/silver/silver_product_category_name_translation


**test**

In [68]:
!spark-submit /opt/spark/etl_pipeline/processing/bronze_to_silver.py --table all

26/04/07 13:56:37 INFO SparkContext: Running Spark version 3.5.8
26/04/07 13:56:37 INFO SparkContext: OS info Linux, 6.6.87.2-microsoft-standard-WSL2, amd64
26/04/07 13:56:37 INFO SparkContext: Java version 11.0.29
26/04/07 13:56:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/07 13:56:37 INFO ResourceUtils: ==============================================================
26/04/07 13:56:37 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/07 13:56:37 INFO ResourceUtils: ==============================================================
26/04/07 13:56:37 INFO SparkContext: Submitted application: SilverLayer-All
26/04/07 13:56:37 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resour

**validation**

In [71]:
from config.settings import S3_SILVER

silver_tables = list_layer_datasets(spark, S3_SILVER)

for table in silver_tables:
    print(table)

s3a://olist-lakehouse-2026/silver/silver_olist_customers_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_geolocation_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_items_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_payments_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_order_reviews_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_orders_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_products_dataset
s3a://olist-lakehouse-2026/silver/silver_olist_sellers_dataset
s3a://olist-lakehouse-2026/silver/silver_product_category_name_translation


In [4]:
from etl_pipeline.utils.s3_reader import read_delta_table

# The exact 9 table names in your Silver layer
silver_tables = [
    "silver_olist_orders_dataset",
    "silver_olist_order_items_dataset",
    "silver_olist_customers_dataset",
    "silver_olist_geolocation_dataset",
    "silver_olist_order_payments_dataset",
    "silver_olist_order_reviews_dataset",
    "silver_olist_products_dataset",
    "silver_olist_sellers_dataset",
    "silver_product_category_name_translation"
]

print("Final Row Counts for Silver Layer:")
print("-" * 60)

for table_name in silver_tables:
    try:
        # Load the table
        df = read_delta_table(spark, "silver", table_name)
        
        # Count the rows
        row_count = df.count()
        
        # Print nicely formatted (left-aligned table name, comma-separated numbers)
        print(f"{table_name:<45} | {row_count:>10,} rows")
    except Exception as e:
        print(f"{table_name:<45} | FAILED TO LOAD: {e}")
        
print("-" * 60)

Final Row Counts for Silver Layer:
------------------------------------------------------------


26/04/15 08:52:14 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/04/15 08:52:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

silver_olist_orders_dataset                   |     99,441 rows


silver_olist_order_items_dataset              |    112,650 rows


silver_olist_customers_dataset                |     99,441 rows


...

silver_olist_order_reviews_dataset            |     98,410 rows


silver_olist_products_dataset                 |     32,951 rows


silver_olist_sellers_dataset                  |      3,095 rows


[Stage 48:================================>                         (5 + 4) / 9]

silver_product_category_name_translation      |         71 rows
------------------------------------------------------------


In [8]:
spark.stop()

In [27]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test_Silver").getOrCreate()

# Đọc bảng Delta Orders ở tầng Silver lên
df_silver = spark.read.format("delta").load("s3a://olist-lakehouse-2026/silver/silver_olist_orders_dataset")

# 1. Kiểm tra Schema xem đã ép kiểu Timestamp thành công chưa?
df_silver.printSchema()

# 2. Kiểm tra xem file chạy có ghi ra đủ số dòng không?
print(f"Tổng số đơn hàng: {df_silver.count()}")

# 3. Xem thử 5 dòng đầu
df_silver.show(5, truncate=False, vertical=True)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



Tổng số đơn hàng: 99441


[Stage 93:>                                                         (0 + 1) / 1]

-RECORD 0---------------------------------------------------------
 order_id                      | a4de5e12d0e969340cb9f32237c16772 
 customer_id                   | 6046d842a9b5a6de033a21980a114490 
 order_status                  | delivered                        
 order_purchase_timestamp      | 2018-06-15 15:44:21              
 order_approved_at             | 2018-06-15 16:00:22              
 order_delivered_carrier_date  | 2018-06-18 17:22:00              
 order_delivered_customer_date | 2018-06-25 22:27:56              
 order_estimated_delivery_date | 2018-07-11 00:00:00              
-RECORD 1---------------------------------------------------------
 order_id                      | 142721e620ef263e3d3ffa99e7c0e096 
 customer_id                   | 3a79d210106a6f0153ba75c77d71392d 
 order_status                  | delivered                        
 order_purchase_timestamp      | 2017-11-24 13:35:51              
 order_approved_at             | 2017-11-24 16:14:35          